# Luka Dončić — XGBoost Points Predictor

Predicts Luka's points per game using only pre-game information (no leakage).

**Features:**
- `roll5_pts` — rolling 5-game avg of prior points
- `roll10_pts` — rolling 10-game avg (captures longer trend)
- `HOME` — home/away flag
- `days_rest` — days since last game
- `opp_def_rating` — opponent defensive rating that season
- `age_at_game` — Luka's age in decimal years
- `season_num` — integer season index (2018=0, 2024=6)

In [ ]:
import glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nba_api.stats.static import teams as nba_teams

files = sorted(glob.glob('../data/raw/luka_*.csv'))
dfs = []
for f in files:
    year = re.search(r'(\d{4}_\d{2})', f).group(1)
    df = pd.read_csv(f)
    df = df[pd.to_numeric(df['Rk'], errors='coerce').notna()].copy()
    df.rename(columns={df.columns[5]: 'HOME_AWAY'}, inplace=True)
    df['HOME']   = (df['HOME_AWAY'] != '@').astype(int)
    df['SEASON'] = year
    df['Date']   = pd.to_datetime(df['Date'])
    for col in ['PTS', 'FGA', 'FG%', '3PA', 'TRB', 'AST']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True).sort_values('Date').reset_index(drop=True)
print(f'Total games loaded: {len(df_all)}')
print(df_all.groupby('SEASON')['PTS'].mean().round(1))

In [ ]:
# Load cached defensive ratings (fetched from nba_api in the linear model notebook)
with open('../data/processed/def_ratings.json') as f:
    raw = json.load(f)
def_rating_lookup = {s: {int(k): v for k, v in d.items()} for s, d in raw.items()}

_static        = nba_teams.get_teams()
NBA_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in _static}
BBREF_TO_NBA   = {'CHO': 'CHA', 'PHO': 'PHX', 'BRK': 'BKN'}

def get_def_rating(row):
    nba_abbr = BBREF_TO_NBA.get(row['Opp'], row['Opp'])
    team_id  = NBA_ABBR_TO_ID.get(nba_abbr)
    if team_id is None:
        return np.nan
    return def_rating_lookup.get(row['SEASON'], {}).get(team_id, np.nan)

print('Defensive ratings loaded.')

In [ ]:
LUKA_DOB = pd.Timestamp('1999-02-28')
df = df_all.copy()

# Lag features — shift(1) ensures no current-game leakage
df['roll5_pts']  = df['PTS'].shift(1).rolling(5,  min_periods=3).mean()
df['roll10_pts'] = df['PTS'].shift(1).rolling(10, min_periods=5).mean()

df['days_rest']      = df['Date'].diff().dt.days.clip(upper=14).fillna(2)
df['opp_def_rating'] = df.apply(get_def_rating, axis=1)
df['age_at_game']    = (df['Date'] - LUKA_DOB).dt.days / 365.25

# season_num is dropped — it is perfectly correlated with age_at_game.
# Having both caused the model to be dominated by a time trend
# ("later season = more points") which breaks badly on the test period
# where Luka's scoring actually dropped after the Lakers trade.

FEATURES = ['roll5_pts', 'roll10_pts', 'HOME', 'days_rest',
            'opp_def_rating', 'age_at_game']

df.dropna(subset=FEATURES + ['PTS'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Model-ready games: {len(df)}')
df[FEATURES + ['PTS']].describe().round(2)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

X = df[FEATURES].values
y = df['PTS'].values

split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {split_idx} games  up to {df["Date"].iloc[split_idx-1].date()}')
print(f'Test : {len(df)-split_idx} games  from {df["Date"].iloc[split_idx].date()}')

## Hyperparameters — edit and re-run

| Parameter | What it controls | Good range |
|---|---|---|
| `n_estimators` | Number of trees. More = more powerful but risks overfitting | 100–1000 |
| `max_depth` | Max depth per tree. Deeper = more complex patterns, more overfit risk | 2–8 |
| `learning_rate` | How much each tree contributes. Lower needs more trees but is more robust | 0.01–0.3 |
| `subsample` | Fraction of rows per tree. Below 1 adds randomness, reduces overfitting | 0.5–1.0 |
| `colsample_bytree` | Fraction of features per tree. Like `subsample` but for columns | 0.5–1.0 |
| `min_child_weight` | Min samples required in a leaf. Higher = more conservative splits | 1–10 |
| `reg_alpha` | L1 regularisation — drives small weights toward zero | 0–1 |
| `reg_lambda` | L2 regularisation — shrinks all weights evenly | 0–5 |

**Tuning tips:**
- Train RMSE much lower than CV RMSE → overfitting → raise `min_child_weight`, lower `max_depth`
- Both RMSEs high → underfitting → more trees, lower `learning_rate`, deeper trees

In [ ]:
# ── Edit these and re-run ────────────────────────────────────────────────────
PARAMS = dict(
    n_estimators     = 200,   # fewer trees → less memorisation
    max_depth        = 2,     # shallow trees → simpler patterns
    learning_rate    = 0.05,
    subsample        = 0.7,
    colsample_bytree = 0.7,
    min_child_weight = 10,    # require more data per leaf → conservative splits
    reg_alpha        = 0.5,   # stronger L1
    reg_lambda       = 2.0,   # stronger L2
    objective        = 'reg:squarederror',
    random_state     = 42,
)
# ─────────────────────────────────────────────────────────────────────────────

model = xgb.XGBRegressor(**PARAMS)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred))

print('=== Metrics ===')
print(f'  Train RMSE : {train_rmse:.2f} pts')
print(f'  Test  R2   : {r2_score(y_test, y_pred):.3f}')
print(f'  Test  RMSE : {test_rmse:.2f} pts')
print(f'  Test  MAE  : {mean_absolute_error(y_test, y_pred):.2f} pts')

tscv = TimeSeriesSplit(n_splits=5)
cv_rmse = [
    np.sqrt(mean_squared_error(
        y[val], xgb.XGBRegressor(**PARAMS).fit(X[tr], y[tr]).predict(X[val])
    ))
    for tr, val in tscv.split(X)
]
print(f'  CV RMSE (5-fold): {np.mean(cv_rmse):.2f} +/- {np.std(cv_rmse):.2f} pts')
print(f'  Train/test gap  : {abs(train_rmse - test_rmse):.2f} pts',
      '(overfitting)' if train_rmse < test_rmse - 1 else '(looks stable)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors='none', color='steelblue', s=40)
lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect')
ax.set_xlabel('Actual PTS'); ax.set_ylabel('Predicted PTS')
ax.set_title('Actual vs Predicted (XGBoost)'); ax.legend()

ax = axes[1]
residuals = y_test - y_pred
ax.scatter(y_pred, residuals, alpha=0.5, edgecolors='none', color='darkorange', s=40)
ax.axhline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted PTS'); ax.set_ylabel('Residual (actual - pred)')
ax.set_title('Residuals vs Predicted')

ax = axes[2]
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance (gain)')
ax.set_xlabel('Importance score')

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs actual over time across the test period
test_dates = df['Date'].iloc[split_idx:].values

plt.figure(figsize=(14, 4))
plt.plot(test_dates, y_test, alpha=0.5, label='Actual', color='steelblue')
plt.plot(test_dates, y_pred, alpha=0.8, label='Predicted', color='darkorange', linestyle='--')
plt.axhline(y_test.mean(), color='grey', linestyle=':', lw=1,
            label=f'Test mean ({y_test.mean():.1f} pts)')
plt.title('XGBoost — Predicted vs Actual PTS (test period)')
plt.xlabel('Date'); plt.ylabel('Points')
plt.legend(); plt.tight_layout()
plt.show()

# Enhanced Feature Set — Rivals + Additional Context

## New features
- **`rival_avg_form`** — average rolling 5-game points of Luka's 4 season rivals, computed up to but not including his game date. Tests whether rival recent scoring predicts Luka's output (competition effect).
- **`b2b`** — 1 if this is the second night of a back-to-back (days_rest == 1). Fatigue signal.
- **`roll5_ast`** — rolling 5-game avg assists. If assists are up, usage may be redistributed away from scoring.
- **`roll5_tov`** — rolling 5-game avg turnovers. High recent turnovers may signal Luka is out of rhythm.

In [ ]:
import time
from nba_api.stats.endpoints import playergamelogs

# Season rivals — 4 players per season who competed most directly with Luka
# Format: {bbref_season_key: [player_id, ...]}
RIVALS = {
    '2018_19': [1629027, 2544,   201935, 1626164],  # Trae, LeBron, Harden, Booker
    '2019_20': [1629027, 2544,   201935, 202695],   # Trae, LeBron, Harden, Kawhi
    '2020_21': [202695,  203999, 203507, 201939],   # Kawhi, Jokic, Giannis, Curry
    '2021_22': [203999,  203507, 201939, 1628369],  # Jokic, Giannis, Curry, Tatum
    '2022_23': [203999,  203507, 1628369, 203954],  # Jokic, Giannis, Tatum, Embiid
    '2023_24': [203999,  1628983, 1628369, 203507], # Jokic, SGA, Tatum, Giannis
    '2024_25': [1628983, 203999, 203507, 1628369],  # SGA, Jokic, Giannis, Tatum
}

# BBRef season key -> nba_api season string
SEASON_STR = {
    '2018_19': '2018-19', '2019_20': '2019-20', '2020_21': '2020-21',
    '2021_22': '2021-22', '2022_23': '2022-23', '2023_24': '2023-24',
    '2024_25': '2024-25',
}

# Build set of unique (player_id, season) pairs to fetch
to_fetch = set()
for season_key, player_ids in RIVALS.items():
    for pid in player_ids:
        to_fetch.add((pid, season_key))

print(f"Fetching {len(to_fetch)} player-season combinations...")

# rival_logs: {player_id: {season_key: DataFrame with Date + PTS}}
rival_logs = {}

for pid, season_key in sorted(to_fetch):
    logs = playergamelogs.PlayerGameLogs(
        player_id_nullable=pid,
        season_nullable=SEASON_STR[season_key],
    ).get_data_frames()[0]

    mini = logs[['GAME_DATE', 'PTS']].copy()
    mini['GAME_DATE'] = pd.to_datetime(mini['GAME_DATE'])
    mini = mini.sort_values('GAME_DATE').reset_index(drop=True)

    if pid not in rival_logs:
        rival_logs[pid] = {}
    rival_logs[pid][season_key] = mini

    print(f"  Fetched player {pid} | {season_key} — {len(mini)} games")
    time.sleep(0.6)

print("\nDone.")

# Save to disk so we never need to hit the API again
save_data = {
    str(pid): {
        sk: df[['GAME_DATE', 'PTS']].assign(GAME_DATE=df['GAME_DATE'].dt.strftime('%Y-%m-%d')).to_dict('records')
        for sk, df in season_dict.items()
    }
    for pid, season_dict in rival_logs.items()
}
with open('../data/processed/rival_logs.json', 'w') as f:
    json.dump(save_data, f)
print("Saved rival_logs.json")

In [ ]:
# Load rival logs from cache (run this instead of the fetch cell on future sessions)
with open('../data/processed/rival_logs.json') as f:
    raw_rivals = json.load(f)

rival_logs = {
    int(pid): {
        sk: pd.DataFrame(records).assign(GAME_DATE=lambda d: pd.to_datetime(d['GAME_DATE']))
        for sk, records in season_dict.items()
    }
    for pid, season_dict in raw_rivals.items()
}
print("Loaded rival_logs.json")

In [ ]:
def rival_rolling_avg(game_date, season_key, window=5):
    """
    For a given game date and season, compute the average of each rival's
    rolling 5-game points total up to (but not including) game_date.
    Returns the mean across all 4 rivals. NaN if insufficient data.
    """
    rival_avgs = []
    for pid in RIVALS.get(season_key, []):
        logs_df = rival_logs.get(pid, {}).get(season_key)
        if logs_df is None:
            continue
        prior = logs_df[logs_df['GAME_DATE'] < game_date]['PTS']
        if len(prior) >= 3:
            rival_avgs.append(prior.iloc[-window:].mean())
    return np.mean(rival_avgs) if rival_avgs else np.nan

df_v3 = df.copy()

# ── Rival avg form ───────────────────────────────────────────────────────────
df_v3['rival_avg_form'] = [
    rival_rolling_avg(row['Date'], row['SEASON'])
    for _, row in df_v3.iterrows()
]

# ── Back-to-back flag ────────────────────────────────────────────────────────
df_v3['b2b'] = (df_v3['Date'].diff().dt.days == 1).astype(int)

# ── Rolling assist and turnover averages (lagged, no leakage) ────────────────
for col, alias in [('AST', 'roll5_ast'), ('TOV', 'roll5_tov')]:
    df_v3[col] = pd.to_numeric(df_v3[col], errors='coerce')
    df_v3[alias] = df_v3[col].shift(1).rolling(5, min_periods=3).mean()

FEATURES_V3 = FEATURES + ['rival_avg_form', 'b2b', 'roll5_ast', 'roll5_tov']

df_v3.dropna(subset=FEATURES_V3 + ['PTS'], inplace=True)
df_v3.reset_index(drop=True, inplace=True)

print(f"Games with all features: {len(df_v3)}")
print(f"  Back-to-back games: {df_v3['b2b'].sum()}")
print(f"  Rival avg form range: {df_v3['rival_avg_form'].min():.1f} – {df_v3['rival_avg_form'].max():.1f} pts")
df_v3[['rival_avg_form', 'b2b', 'roll5_ast', 'roll5_tov']].describe().round(2)

In [ ]:
X3 = df_v3[FEATURES_V3].values
y3 = df_v3['PTS'].values

split_idx3 = int(len(df_v3) * 0.8)
X3_train, X3_test = X3[:split_idx3], X3[split_idx3:]
y3_train, y3_test = y3[:split_idx3], y3[split_idx3:]

print(f"Train: {split_idx3} games  →  up to {df_v3['Date'].iloc[split_idx3-1].date()}")
print(f"Test : {len(df_v3)-split_idx3} games  →  from {df_v3['Date'].iloc[split_idx3].date()}")

model_v3 = xgb.XGBRegressor(**PARAMS)
model_v3.fit(X3_train, y3_train)
y3_pred = model_v3.predict(X3_test)

train_rmse3 = np.sqrt(mean_squared_error(y3_train, model_v3.predict(X3_train)))
test_rmse3  = np.sqrt(mean_squared_error(y3_test, y3_pred))

print("\n=== V3 Metrics (with rivals + b2b + ast + tov) ===")
print(f"  Train RMSE : {train_rmse3:.2f} pts")
print(f"  Test  R2   : {r2_score(y3_test, y3_pred):.3f}")
print(f"  Test  RMSE : {test_rmse3:.2f} pts")
print(f"  Test  MAE  : {mean_absolute_error(y3_test, y3_pred):.2f} pts")

tscv3 = TimeSeriesSplit(n_splits=5)
cv3 = [
    np.sqrt(mean_squared_error(
        y3[val], xgb.XGBRegressor(**PARAMS).fit(X3[tr], y3[tr]).predict(X3[val])
    ))
    for tr, val in tscv3.split(X3)
]
print(f"  CV RMSE (5-fold): {np.mean(cv3):.2f} +/- {np.std(cv3):.2f} pts")
print(f"  Train/test gap  : {abs(train_rmse3 - test_rmse3):.2f} pts",
      '(overfitting)' if train_rmse3 < test_rmse3 - 1 else '(looks stable)')

# ── Feature importance — all features ranked ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y3_test, y3_pred, alpha=0.5, edgecolors='none', color='steelblue', s=40)
lims = [min(y3_test.min(), y3_pred.min()) - 2, max(y3_test.max(), y3_pred.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect')
ax.set_xlabel('Actual PTS'); ax.set_ylabel('Predicted PTS')
ax.set_title('Actual vs Predicted (V3 — with rivals)'); ax.legend()

ax = axes[1]
imp = pd.Series(model_v3.feature_importances_, index=FEATURES_V3).sort_values()
colors = ['crimson' if f in ['rival_avg_form', 'b2b', 'roll5_ast', 'roll5_tov']
          else 'steelblue' for f in imp.index]
imp.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Importance — red = new features')
ax.set_xlabel('Importance score')

plt.tight_layout()
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(model_v3)
shap_values = explainer.shap_values(X3_train)  # use training set for stable estimates

# ── Beeswarm plot ────────────────────────────────────────────────────────────
# Each dot = one game.
# X-axis = SHAP value: how much that feature pushed the prediction up (+) or down (-).
# Colour = feature value: red = high, blue = low.
# This shows both direction and magnitude for every feature across every game.
shap.summary_plot(
    shap_values,
    X3_train,
    feature_names=FEATURES_V3,
    plot_type='dot',
    show=True,
)

# ── Bar plot (mean absolute SHAP — cleaner importance ranking) ───────────────
shap.summary_plot(
    shap_values,
    X3_train,
    feature_names=FEATURES_V3,
    plot_type='bar',
    show=True,
)

In [ ]:
df_v4 = df_v3.copy()

# ── Replace age_at_game with season_avg_so_far ───────────────────────────────
# For each game, compute Luka's expanding mean points within that season up
# to (but not including) the current game. This tells the model "what kind
# of season is Luka having?" without encoding a career-long upward trend.
# Shift by 1 so the current game's score is not included (no leakage).
df_v4['season_avg_so_far'] = (
    df_v4.groupby('SEASON')['PTS']
    .transform(lambda s: s.shift(1).expanding().mean())
)

# Also add relative form: how hot/cold vs his own season baseline
# Positive = running above his season avg, negative = below
df_v4['form_vs_season'] = df_v4['roll5_pts'] - df_v4['season_avg_so_far']

# Drop age_at_game and days_rest (SHAP showed days_rest is near zero)
FEATURES_V4 = [f for f in FEATURES_V3 if f not in ('age_at_game', 'days_rest')]
FEATURES_V4 += ['season_avg_so_far', 'form_vs_season']

df_v4.dropna(subset=FEATURES_V4 + ['PTS'], inplace=True)
df_v4.reset_index(drop=True, inplace=True)

print(f"Features: {FEATURES_V4}")
print(f"Games: {len(df_v4)}")
df_v4[['season_avg_so_far', 'form_vs_season']].describe().round(2)

In [ ]:
X4 = df_v4[FEATURES_V4].values
y4 = df_v4['PTS'].values

split_idx4 = int(len(df_v4) * 0.8)
X4_train, X4_test = X4[:split_idx4], X4[split_idx4:]
y4_train, y4_test = y4[:split_idx4], y4[split_idx4:]

print(f"Train: {split_idx4} games  →  up to {df_v4['Date'].iloc[split_idx4-1].date()}")
print(f"Test : {len(df_v4)-split_idx4} games  →  from {df_v4['Date'].iloc[split_idx4].date()}")

model_v4 = xgb.XGBRegressor(**PARAMS)
model_v4.fit(X4_train, y4_train)
y4_pred = model_v4.predict(X4_test)

train_rmse4 = np.sqrt(mean_squared_error(y4_train, model_v4.predict(X4_train)))
test_rmse4  = np.sqrt(mean_squared_error(y4_test, y4_pred))

print("\n=== V4 Metrics (season-centred form, no age_at_game) ===")
print(f"  Train RMSE : {train_rmse4:.2f} pts")
print(f"  Test  R2   : {r2_score(y4_test, y4_pred):.3f}")
print(f"  Test  RMSE : {test_rmse4:.2f} pts")
print(f"  Test  MAE  : {mean_absolute_error(y4_test, y4_pred):.2f} pts")

tscv4 = TimeSeriesSplit(n_splits=5)
cv4 = [
    np.sqrt(mean_squared_error(
        y4[val], xgb.XGBRegressor(**PARAMS).fit(X4[tr], y4[tr]).predict(X4[val])
    ))
    for tr, val in tscv4.split(X4)
]
print(f"  CV RMSE (5-fold): {np.mean(cv4):.2f} +/- {np.std(cv4):.2f} pts")
print(f"  Train/test gap  : {abs(train_rmse4 - test_rmse4):.2f} pts",
      '(overfitting)' if train_rmse4 < test_rmse4 - 1 else '(looks stable)')

# ── SHAP beeswarm ────────────────────────────────────────────────────────────
explainer4    = shap.TreeExplainer(model_v4)
shap_values4  = explainer4.shap_values(X4_train)

shap.summary_plot(shap_values4, X4_train, feature_names=FEATURES_V4,
                  plot_type='dot', show=True)

In [ ]:
test_dates4 = df_v4['Date'].iloc[split_idx4:].values
residuals4  = y4_test - y4_pred

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Top: predicted vs actual ─────────────────────────────────────────────────
ax = axes[0]
ax.plot(test_dates4, y4_test, alpha=0.5, color='steelblue', label='Actual')
ax.plot(test_dates4, y4_pred, alpha=0.8, color='darkorange',
        linestyle='--', label='Predicted')
ax.axhline(y4_test.mean(), color='grey', linestyle=':', lw=1,
           label=f'Test mean ({y4_test.mean():.1f} pts)')
ax.set_title('V4 — Predicted vs Actual PTS (test period)')
ax.set_ylabel('Points')
ax.legend()

# ── Bottom: residuals — scatter avoids the datetime bar-width issue ──────────
ax = axes[1]
colors = ['steelblue' if r >= 0 else 'crimson' for r in residuals4]
ax.scatter(test_dates4, residuals4, c=colors, alpha=0.7, s=30, zorder=3)
ax.vlines(test_dates4, 0, residuals4, colors=colors, alpha=0.4, linewidth=1)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Residuals over time  (blue = under-predicted, red = over-predicted)')
ax.set_xlabel('Date')
ax.set_ylabel('Residual (actual − pred)')

plt.tight_layout()
plt.show()

# Walk-Forward Validation + Quantile Regression

## Walk-forward validation
Instead of one fixed 80/20 split, we expand the training window month by month and test on the next month's games. This simulates real-world usage — at any point in time, the model only sees past data — and gives a more honest RMSE across many different test windows.

## Quantile regression
Instead of predicting a single point estimate ("Luka will score 32 pts"), we train three models predicting the 10th, 50th and 90th percentiles. This gives an 80% prediction interval ("Luka will score between 24 and 40 pts") which is more honest and more useful in practice.

In [ ]:
from sklearn.metrics import mean_squared_error
import calendar

# Walk-forward: expand training by one month, test on the next
# Minimum 150 games (~2 seasons) before we start testing
MIN_TRAIN = 150

dates      = df_v4['Date']
X_wf       = df_v4[FEATURES_V4].values
y_wf       = df_v4['PTS'].values

# Get all year-month pairs in chronological order
months = sorted(dates.dt.to_period('M').unique())

wf_results = []   # list of (date, actual, predicted)

for i, test_month in enumerate(months):
    # All games strictly before this month = training set
    train_mask = dates.dt.to_period('M') < test_month
    test_mask  = dates.dt.to_period('M') == test_month

    if train_mask.sum() < MIN_TRAIN or test_mask.sum() == 0:
        continue

    X_tr, y_tr = X_wf[train_mask], y_wf[train_mask]
    X_te, y_te = X_wf[test_mask],  y_wf[test_mask]

    m = xgb.XGBRegressor(**PARAMS)
    m.fit(X_tr, y_tr)
    preds = m.predict(X_te)

    for date, actual, pred in zip(dates[test_mask], y_te, preds):
        wf_results.append({'date': date, 'actual': actual, 'pred': pred,
                           'month': str(test_month)})

wf_df = pd.DataFrame(wf_results)
wf_df['residual'] = wf_df['actual'] - wf_df['pred']

overall_rmse = np.sqrt(mean_squared_error(wf_df['actual'], wf_df['pred']))
overall_mae  = np.mean(np.abs(wf_df['residual']))
overall_r2   = r2_score(wf_df['actual'], wf_df['pred'])

print(f"Walk-forward results: {len(wf_df)} games tested across {wf_df['month'].nunique()} months")
print(f"  RMSE : {overall_rmse:.2f} pts")
print(f"  MAE  : {overall_mae:.2f} pts")
print(f"  R²   : {overall_r2:.3f}")

# Monthly RMSE breakdown
monthly_rmse = (
    wf_df.groupby('month')
    .apply(lambda g: np.sqrt(mean_squared_error(g['actual'], g['pred'])))
    .reset_index(name='rmse')
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(wf_df['date'], wf_df['actual'], alpha=0.4, color='steelblue', label='Actual')
ax.plot(wf_df['date'], wf_df['pred'],   alpha=0.8, color='darkorange',
        linestyle='--', label='Predicted (walk-forward)')
ax.axhline(wf_df['actual'].mean(), color='grey', linestyle=':', lw=1,
           label=f"Mean ({wf_df['actual'].mean():.1f} pts)")
ax.set_title('Walk-Forward Predictions vs Actual')
ax.set_ylabel('Points')
ax.legend()

ax = axes[1]
ax.bar(range(len(monthly_rmse)), monthly_rmse['rmse'], color='steelblue', alpha=0.7)
ax.axhline(overall_rmse, color='red', linestyle='--', lw=1.5,
           label=f'Overall RMSE ({overall_rmse:.2f})')
ax.set_xticks(range(len(monthly_rmse)))
ax.set_xticklabels(monthly_rmse['month'], rotation=45, ha='right', fontsize=7)
ax.set_title('RMSE per Month (walk-forward)')
ax.set_ylabel('RMSE (pts)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Quantile regression — train three models for 10th, 50th, 90th percentile
# XGBoost natively supports quantile regression via objective='reg:quantileerror'
# This gives us an 80% prediction interval (between the 10th and 90th percentile)

q_params = {**PARAMS, 'objective': 'reg:quantileerror'}

model_q10 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.10})
model_q50 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.50})
model_q90 = xgb.XGBRegressor(**{**q_params, 'quantile_alpha': 0.90})

for m in (model_q10, model_q50, model_q90):
    m.fit(X4_train, y4_train)

pred_q10 = model_q10.predict(X4_test)
pred_q50 = model_q50.predict(X4_test)
pred_q90 = model_q90.predict(X4_test)

# Coverage: what % of actual values fall within the 80% interval?
within_interval = ((y4_test >= pred_q10) & (y4_test <= pred_q90))
coverage = within_interval.mean() * 100
avg_width = (pred_q90 - pred_q10).mean()

print("=== Quantile Regression (80% prediction interval) ===")
print(f"  Median (q50) RMSE : {np.sqrt(mean_squared_error(y4_test, pred_q50)):.2f} pts")
print(f"  Coverage          : {coverage:.1f}%  (target: 80%)")
print(f"  Avg interval width: {avg_width:.1f} pts")
print(f"  e.g. 'Luka will score between X and Y pts (80% confidence)'")

# ── Plot: prediction interval on test set ────────────────────────────────────
test_dates_q = df_v4['Date'].iloc[split_idx4:].values

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(test_dates_q, pred_q10, pred_q90,
                alpha=0.25, color='darkorange', label='80% prediction interval')
ax.plot(test_dates_q, pred_q50, color='darkorange', lw=1.5,
        linestyle='--', label='Predicted median (q50)')
ax.scatter(test_dates_q, y4_test, color='steelblue', s=20, alpha=0.7,
           zorder=3, label='Actual')

# Highlight games that fell outside the interval
outside = ~within_interval
ax.scatter(test_dates_q[outside], y4_test[outside],
           color='red', s=40, zorder=4, label=f'Outside interval ({outside.sum()} games)')

ax.set_title(f'Quantile Regression — 80% Prediction Interval  |  Coverage: {coverage:.1f}%')
ax.set_xlabel('Date')
ax.set_ylabel('Points')
ax.legend()
plt.tight_layout()
plt.show()

# Model Assessment — Critical Analysis

---

## Verdict: Promising proof-of-concept, not production-ready

The model captures real signals and tells a coherent story about what drives Luka's scoring. But the headline metrics — **RMSE ~9 pts on a player averaging 29 pts (~31% error)**, **coverage 75% vs a target 80%** — make it unsuitable for precise prediction. This section breaks down exactly why, what is genuinely working, and what the path forward looks like.

---

## What actually works

### 1. The competition effect is statistically real
`rival_avg_form` consistently appeared in the top half of feature importance across all model versions. The SHAP plot showed it has meaningful impact, though the direction is non-monotonic (not simply "hot rivals = more Luka pts"). There is something here worth investigating further with more granular opponent-level data.

### 2. The scoring/playmaking trade-off is detectable
`roll5_ast` was one of the strongest new features (SHAP ~1.05), and the beeswarm confirmed the correct direction: when Luka's assists are high, the model predicts fewer points. This is a real basketball phenomenon — playmaker mode and scorer mode compete for usage — and the model learned it from data alone.

### 3. Back-to-back fatigue is measurable
`b2b` showed a clean, consistent negative SHAP value. Being on the second night of a back-to-back genuinely suppresses Luka's output. This is one of the clearest and most trustworthy signals in the model.

### 4. Opponent defensive quality matters (weakly)
`opp_def_rating` is in the right direction — harder defences suppress scoring — but the signal is noisy. A season-level defensive rating is too coarse; what matters is how the defence performs in the specific matchup context.

---

## What is not working

### 1. The dataset is too small for XGBoost
405 games across 7 seasons is tiny for a tree ensemble. XGBoost is designed for datasets with tens of thousands of rows. With 405 rows and 11 features, it memorises rather than generalises. The persistent train/test RMSE gap (3–6 pts) across all versions is direct evidence of this. A linear model with these features would likely achieve similar test RMSE with far less complexity.

### 2. The model cannot handle structural breaks
The quantile plot makes this explicit. After the Lakers trade (Feb 2025), 20 of 81 test games fall outside the 80% interval — the interval is clustered around Dallas-era scoring levels when Luka is now in a different offensive system averaging ~5 fewer points per game. No amount of feature engineering fixes this without retraining after the trade. A model trained purely on post-trade data would almost certainly outperform this one for 2025 predictions.

### 3. The prediction interval is too wide to be useful
An average interval width of **19.3 pts** — roughly "Luka will score between 21 and 40 pts" — covers too much of the realistic scoring range to support actionable decisions. For context, the interquartile range of his actual scoring is 23–35 pts (12 pts). Our interval is wider than the interquartile range, meaning we are not providing useful information beyond "he'll probably score somewhere in his normal range."

### 4. Coverage is 75%, not 80%
We are missing the calibration target by 5 percentage points. The model is systematically overconfident — its intervals are too narrow in the tails. The 20 red dots cluster in two places: explosive outlier games (50+ pts) where no model will ever have a wide enough upper bound, and the post-trade period where the lower bound is consistently too high. Both are systematic, not random.

### 5. `season_avg_so_far` and `age_at_game` remain dominant
Despite dropping `age_at_game` in V4 and replacing it with `season_avg_so_far`, the model still leans heavily on a time-trend proxy. The dominant prediction for any given game is still essentially "what is Luka's scoring level this season?" — not "what will happen in this specific game." Game-specific features (opponent, rest, rivals) are secondary adjustments around a seasonal baseline.

---

## Quantile regression assessment

| Metric | Result | Interpretation |
|---|---|---|
| Median RMSE | 9.12 pts | ~31% of his scoring average — a wide miss for precise prediction |
| Coverage | 75% | 5% below target — model is overconfident |
| Interval width | 19.3 pts | Wider than his IQR — limited practical value |
| Games outside interval | 20/81 | Clustered around outliers and the post-trade period |

The quantile framework itself is the right approach — outputting a range is more honest than a point estimate. The problem is the underlying model, not the quantile wrapping.

---

## Walk-forward assessment

Walk-forward RMSE is the most honest metric we have produced. Unlike the single 80/20 split (which could be lucky or unlucky depending on where the cut falls), walk-forward averages performance across every monthly window. The monthly RMSE bar chart reveals which periods are hardest to predict — expect spikes around season transitions, injury-return games, and the Lakers trade period.

---

## If this were a real project — what next?

| Priority | Action | Expected impact |
|---|---|---|
| **High** | Expand to 20–30 star players | 10× more data; model learns general patterns not Luka-specific quirks |
| **High** | Retrain separately on post-trade data | Eliminates the structural break problem for near-term predictions |
| **Medium** | Add opponent pace (possessions per game) | More possessions = more scoring chances; stronger than defensive rating |
| **Medium** | Add per-game usage rate (rolling) | Captures offensive load more directly than assists or minutes |
| **Low** | Hyperparameter search with Optuna | Likely marginal gains; data volume is the real constraint |
| **Low** | Neural network (LSTM) | Only worthwhile with significantly more data; overkill here |

---

## Summary

The model is a solid analytical exercise. It correctly identifies real basketball signals (B2B fatigue, playmaking trade-off, competition effect, home advantage). The architecture — XGBoost with walk-forward validation and quantile output — is the right production setup. What it lacks is data volume and structural break handling. With a multi-player dataset and a trade/team-change indicator, RMSE could realistically drop to the 7–8 pt range, which is close to the theoretical floor for individual game prediction.